In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_folder = "/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025"
%cd "{base_folder}"

/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025


In [3]:
from pathlib import Path
import os
import sqlite3
import pandas as pd

def load_heart_data():
    print("[1] Loading heart.csv from local file…")
    csv_path = Path(f"{base_folder}/data/heart.csv")
    if csv_path.is_file():
        heart_df = pd.read_csv(csv_path)
        print(f"Successfully loaded {len(heart_df)} rows.")
        return heart_df
    else:
        print(f"ERROR: heart.csv not found at {csv_path}")
        print("Please ensure the file exists in the ./data/ directory.")
        exit(1)

def build_3nf_sqlite(db_path="heart.db"):
    print("=== BUILDING 3NF SQLITE DATA MODEL ===")

    print("\n[STEP 1] Loading CSV into DataFrame…")
    heart = load_heart_data()
    print(f"Loaded {len(heart)} rows.")

    print("\n[STEP 2] Creating surrogate key patient_id…")
    heart = heart.reset_index().rename(columns={"index": "patient_id"})
    print("patient_id added.")

    print("\n[STEP 3] Building dimension tables for categorical features…")

    # Chest Pain Type dimension
    cp_dim = heart[["cp"]].drop_duplicates().reset_index(drop=True)
    cp_dim["cp_id"] = cp_dim.index + 1
    cp_dim = cp_dim.rename(columns={"cp": "name"})

    # Resting ECG dimension
    ecg_dim = heart[["restecg"]].drop_duplicates().reset_index(drop=True)
    ecg_dim["ecg_id"] = ecg_dim.index + 1
    ecg_dim = ecg_dim.rename(columns={"restecg": "name"})

    # Slope dimension
    slope_dim = heart[["slope"]].drop_duplicates().reset_index(drop=True)
    slope_dim["slope_id"] = slope_dim.index + 1
    slope_dim = slope_dim.rename(columns={"slope": "name"})

    # Thal dimension
    thal_dim = heart[["thal"]].drop_duplicates().reset_index(drop=True)
    thal_dim["thal_id"] = thal_dim.index + 1
    thal_dim = thal_dim.rename(columns={"thal": "name"})

    print(f"Found {len(cp_dim)} unique chest pain types.")
    print(f"Found {len(ecg_dim)} unique ECG results.")
    print(f"Found {len(slope_dim)} unique slope values.")
    print(f"Found {len(thal_dim)} unique thal values.")

    print("\n[STEP 4] Merging dimension IDs into main DataFrame…")
    heart = heart.merge(cp_dim[["name", "cp_id"]],
                        left_on="cp", right_on="name", how="left")
    heart = heart.drop(columns=["name"])

    heart = heart.merge(ecg_dim[["name", "ecg_id"]],
                        left_on="restecg", right_on="name", how="left")
    heart = heart.drop(columns=["name"])

    heart = heart.merge(slope_dim[["name", "slope_id"]],
                        left_on="slope", right_on="name", how="left")
    heart = heart.drop(columns=["name"])

    heart = heart.merge(thal_dim[["name", "thal_id"]],
                        left_on="thal", right_on="name", how="left")
    heart = heart.drop(columns=["name"])

    print("\n[STEP 5] Creating 3NF DataFrames…")

    # Dimension tables
    df_cp = cp_dim[["cp_id", "name"]]
    df_ecg = ecg_dim[["ecg_id", "name"]]
    df_slope = slope_dim[["slope_id", "name"]]
    df_thal = thal_dim[["thal_id", "name"]]

    # Patient dimension table - REFACTORED: oldpeak removed
    df_patient = heart[
        ["patient_id", "age", "sex", "cp_id", "trestbps", "chol", "fbs",
         "ecg_id", "thalach", "exang", "slope_id", "ca", "thal_id"]
    ].drop_duplicates(subset=["patient_id"])

    # Target/diagnosis table
    df_diagnosis = heart[
        ["patient_id", "target"]
    ]

    print("3NF DataFrames created.")

    print("\n[STEP 6] Creating SQLite database and tables…")
    if os.path.exists(db_path):
        print("Existing DB found. Removing…")
        os.remove(db_path)

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    print("Running SQL schema creation script…")
    cur.executescript(
        """
        DROP TABLE IF EXISTS diagnosis;
        DROP TABLE IF EXISTS patient;
        DROP TABLE IF EXISTS thal;
        DROP TABLE IF EXISTS slope;
        DROP TABLE IF EXISTS ecg;
        DROP TABLE IF EXISTS chest_pain_type;

        -- Chest Pain Type dimension
        CREATE TABLE chest_pain_type (
            cp_id  INTEGER PRIMARY KEY,
            name   TEXT NOT NULL UNIQUE
        );

        -- Resting ECG dimension
        CREATE TABLE ecg (
            ecg_id INTEGER PRIMARY KEY,
            name   TEXT NOT NULL UNIQUE
        );

        -- Slope dimension
        CREATE TABLE slope (
            slope_id INTEGER PRIMARY KEY,
            name     TEXT NOT NULL UNIQUE
        );

        -- Thal dimension
        CREATE TABLE thal (
            thal_id INTEGER PRIMARY KEY,
            name    TEXT NOT NULL UNIQUE
        );

        -- Patient table - REFACTORED: oldpeak column removed
        CREATE TABLE patient (
            patient_id     INTEGER PRIMARY KEY,
            age            INTEGER NOT NULL,
            sex            INTEGER NOT NULL,  -- 1 = male, 0 = female
            cp_id          INTEGER NOT NULL,
            trestbps       INTEGER NOT NULL,  -- resting blood pressure
            chol           INTEGER NOT NULL,  -- cholesterol
            fbs            INTEGER NOT NULL,  -- fasting blood sugar > 120 (1=true, 0=false)
            ecg_id         INTEGER NOT NULL,
            thalach        INTEGER NOT NULL,  -- max heart rate
            exang          INTEGER NOT NULL,  -- exercise induced angina (1=yes, 0=no)
            slope_id       INTEGER NOT NULL,
            ca             INTEGER NOT NULL,  -- number of major vessels
            thal_id        INTEGER NOT NULL,
            FOREIGN KEY (cp_id) REFERENCES chest_pain_type(cp_id),
            FOREIGN KEY (ecg_id) REFERENCES ecg(ecg_id),
            FOREIGN KEY (slope_id) REFERENCES slope(slope_id),
            FOREIGN KEY (thal_id) REFERENCES thal(thal_id)
        );

        -- Diagnosis table
        CREATE TABLE diagnosis (
            patient_id INTEGER PRIMARY KEY,
            target     INTEGER NOT NULL,  -- 0 = no disease, 1 = disease
            FOREIGN KEY (patient_id) REFERENCES patient(patient_id)
        );
        """
    )
    print("Tables created.")

    print("\n[STEP 7] Inserting data into SQLite database…")

    print("Inserting chest_pain_type dimension…")
    cur.executemany(
        "INSERT INTO chest_pain_type (cp_id, name) VALUES (?, ?)",
        list(df_cp.itertuples(index=False, name=None)),
    )

    print("Inserting ecg dimension…")
    cur.executemany(
        "INSERT INTO ecg (ecg_id, name) VALUES (?, ?)",
        list(df_ecg.itertuples(index=False, name=None)),
    )

    print("Inserting slope dimension…")
    cur.executemany(
        "INSERT INTO slope (slope_id, name) VALUES (?, ?)",
        list(df_slope.itertuples(index=False, name=None)),
    )

    print("Inserting thal dimension…")
    cur.executemany(
        "INSERT INTO thal (thal_id, name) VALUES (?, ?)",
        list(df_thal.itertuples(index=False, name=None)),
    )

    print("Inserting patient table…")
    # REFACTORED: oldpeak removed from column list and values
    cur.executemany(
        """
        INSERT INTO patient (
            patient_id, age, sex, cp_id, trestbps, chol, fbs, ecg_id,
            thalach, exang, slope_id, ca, thal_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        list(df_patient.itertuples(index=False, name=None)),
    )

    print("Inserting diagnosis table…")
    cur.executemany(
        "INSERT INTO diagnosis (patient_id, target) VALUES (?, ?)",
        list(df_diagnosis.itertuples(index=False, name=None)),
    )

    conn.commit()
    conn.close()

    print("\n=== DONE! SQLite DB created at:", db_path, "===\n")

# Run the build
build_3nf_sqlite("./data/heart.db")

=== BUILDING 3NF SQLITE DATA MODEL ===

[STEP 1] Loading CSV into DataFrame…
[1] Loading heart.csv from local file…
Successfully loaded 1025 rows.
Loaded 1025 rows.

[STEP 2] Creating surrogate key patient_id…
patient_id added.

[STEP 3] Building dimension tables for categorical features…
Found 4 unique chest pain types.
Found 3 unique ECG results.
Found 3 unique slope values.
Found 4 unique thal values.

[STEP 4] Merging dimension IDs into main DataFrame…

[STEP 5] Creating 3NF DataFrames…
3NF DataFrames created.

[STEP 6] Creating SQLite database and tables…
Existing DB found. Removing…
Running SQL schema creation script…
Tables created.

[STEP 7] Inserting data into SQLite database…
Inserting chest_pain_type dimension…
Inserting ecg dimension…
Inserting slope dimension…
Inserting thal dimension…
Inserting patient table…
Inserting diagnosis table…

=== DONE! SQLite DB created at: ./data/heart.db ===



In [ ]:
from pathlib import Path
import os
import sqlite3
import pandas as pd

def load_heart_data():
    print("[1] Loading heart.csv from local file…")
    csv_path = Path(f"{base_folder}/data/heart.csv")
    if csv_path.is_file():
        heart_df = pd.read_csv(csv_path)
        print(f"Successfully loaded {len(heart_df)} rows.")
        return heart_df
    else:
        print(f"ERROR: heart.csv not found at {csv_path}")
        print("Please ensure the file exists in the ./data/ directory.")
        exit(1)

def build_3nf_sqlite(db_path="heart.db"):
    print("=== BUILDING 3NF SQLITE DATA MODEL ===")

    print("\n[STEP 1] Loading CSV into DataFrame…")
    heart = load_heart_data()
    print(f"Loaded {len(heart)} rows.")

    print("\n[STEP 2] Creating surrogate key patient_id…")
    heart = heart.reset_index().rename(columns={"index": "patient_id"})
    print("patient_id added.")

    print("\n[STEP 3] Building dimension tables for categorical features…")

    # Chest Pain Type dimension
    cp_dim = heart[["cp"]].drop_duplicates().reset_index(drop=True)
    cp_dim["cp_id"] = cp_dim.index + 1
    cp_dim = cp_dim.rename(columns={"cp": "name"})

    # Resting ECG dimension
    ecg_dim = heart[["restecg"]].drop_duplicates().reset_index(drop=True)
    ecg_dim["ecg_id"] = ecg_dim.index + 1
    ecg_dim = ecg_dim.rename(columns={"restecg": "name"})

    # Slope dimension
    slope_dim = heart[["slope"]].drop_duplicates().reset_index(drop=True)
    slope_dim["slope_id"] = slope_dim.index + 1
    slope_dim = slope_dim.rename(columns={"slope": "name"})

    # Thal dimension
    thal_dim = heart[["thal"]].drop_duplicates().reset_index(drop=True)
    thal_dim["thal_id"] = thal_dim.index + 1
    thal_dim = thal_dim.rename(columns={"thal": "name"})

    print(f"Found {len(cp_dim)} unique chest pain types.")
    print(f"Found {len(ecg_dim)} unique ECG results.")
    print(f"Found {len(slope_dim)} unique slope values.")
    print(f"Found {len(thal_dim)} unique thal values.")

    print("\n[STEP 4] Merging dimension IDs into main DataFrame…")
    heart = heart.merge(cp_dim[["name", "cp_id"]],
                        left_on="cp", right_on="name", how="left")
    heart = heart.drop(columns=["name"])

    heart = heart.merge(ecg_dim[["name", "ecg_id"]],
                        left_on="restecg", right_on="name", how="left")
    heart = heart.drop(columns=["name"])

    heart = heart.merge(slope_dim[["name", "slope_id"]],
                        left_on="slope", right_on="name", how="left")
    heart = heart.drop(columns=["name"])

    heart = heart.merge(thal_dim[["name", "thal_id"]],
                        left_on="thal", right_on="name", how="left")
    heart = heart.drop(columns=["name"])

    print("\n[STEP 5] Creating 3NF DataFrames…")

    # Dimension tables
    df_cp = cp_dim[["cp_id", "name"]]
    df_ecg = ecg_dim[["ecg_id", "name"]]
    df_slope = slope_dim[["slope_id", "name"]]
    df_thal = thal_dim[["thal_id", "name"]]

    # Patient dimension table
    df_patient = heart[
        ["patient_id", "age", "sex", "cp_id", "trestbps", "chol", "fbs",
         "ecg_id", "thalach", "exang", "oldpeak", "slope_id", "ca", "thal_id"]
    ].drop_duplicates(subset=["patient_id"])

    # Target/diagnosis table
    df_diagnosis = heart[
        ["patient_id", "target"]
    ]

    print("3NF DataFrames created.")

    print("\n[STEP 6] Creating SQLite database and tables…")
    if os.path.exists(db_path):
        print("Existing DB found. Removing…")
        os.remove(db_path)

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    print("Running SQL schema creation script…")
    cur.executescript(
        """
        DROP TABLE IF EXISTS diagnosis;
        DROP TABLE IF EXISTS patient;
        DROP TABLE IF EXISTS thal;
        DROP TABLE IF EXISTS slope;
        DROP TABLE IF EXISTS ecg;
        DROP TABLE IF EXISTS chest_pain_type;

        -- Chest Pain Type dimension
        CREATE TABLE chest_pain_type (
            cp_id  INTEGER PRIMARY KEY,
            name   TEXT NOT NULL UNIQUE
        );

        -- Resting ECG dimension
        CREATE TABLE ecg (
            ecg_id INTEGER PRIMARY KEY,
            name   TEXT NOT NULL UNIQUE
        );

        -- Slope dimension
        CREATE TABLE slope (
            slope_id INTEGER PRIMARY KEY,
            name     TEXT NOT NULL UNIQUE
        );

        -- Thal dimension
        CREATE TABLE thal (
            thal_id INTEGER PRIMARY KEY,
            name    TEXT NOT NULL UNIQUE
        );

        -- Patient table
        CREATE TABLE patient (
            patient_id     INTEGER PRIMARY KEY,
            age            INTEGER NOT NULL,
            sex            INTEGER NOT NULL,  -- 1 = male, 0 = female
            cp_id          INTEGER NOT NULL,
            trestbps       INTEGER NOT NULL,  -- resting blood pressure
            chol           INTEGER NOT NULL,  -- cholesterol
            fbs            INTEGER NOT NULL,  -- fasting blood sugar > 120 (1=true, 0=false)
            ecg_id         INTEGER NOT NULL,
            thalach        INTEGER NOT NULL,  -- max heart rate
            exang          INTEGER NOT NULL,  -- exercise induced angina (1=yes, 0=no)
            oldpeak        REAL NOT NULL,     -- ST depression
            slope_id       INTEGER NOT NULL,
            ca             INTEGER NOT NULL,  -- number of major vessels
            thal_id        INTEGER NOT NULL,
            FOREIGN KEY (cp_id) REFERENCES chest_pain_type(cp_id),
            FOREIGN KEY (ecg_id) REFERENCES ecg(ecg_id),
            FOREIGN KEY (slope_id) REFERENCES slope(slope_id),
            FOREIGN KEY (thal_id) REFERENCES thal(thal_id)
        );

        -- Diagnosis table
        CREATE TABLE diagnosis (
            patient_id INTEGER PRIMARY KEY,
            target     INTEGER NOT NULL,  -- 0 = no disease, 1 = disease
            FOREIGN KEY (patient_id) REFERENCES patient(patient_id)
        );
        """
    )
    print("Tables created.")

    print("\n[STEP 7] Inserting data into SQLite database…")

    print("Inserting chest_pain_type dimension…")
    cur.executemany(
        "INSERT INTO chest_pain_type (cp_id, name) VALUES (?, ?)",
        list(df_cp.itertuples(index=False, name=None)),
    )

    print("Inserting ecg dimension…")
    cur.executemany(
        "INSERT INTO ecg (ecg_id, name) VALUES (?, ?)",
        list(df_ecg.itertuples(index=False, name=None)),
    )

    print("Inserting slope dimension…")
    cur.executemany(
        "INSERT INTO slope (slope_id, name) VALUES (?, ?)",
        list(df_slope.itertuples(index=False, name=None)),
    )

    print("Inserting thal dimension…")
    cur.executemany(
        "INSERT INTO thal (thal_id, name) VALUES (?, ?)",
        list(df_thal.itertuples(index=False, name=None)),
    )

    print("Inserting patient table…")
    cur.executemany(
        """
        INSERT INTO patient (
            patient_id, age, sex, cp_id, trestbps, chol, fbs, ecg_id,
            thalach, exang, oldpeak, slope_id, ca, thal_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        list(df_patient.itertuples(index=False, name=None)),
    )

    print("Inserting diagnosis table…")
    cur.executemany(
        "INSERT INTO diagnosis (patient_id, target) VALUES (?, ?)",
        list(df_diagnosis.itertuples(index=False, name=None)),
    )

    conn.commit()
    conn.close()

    print("\n=== DONE! SQLite DB created at:", db_path, "===\n")

# Run the build
build_3nf_sqlite("./data/heart.db")

=== BUILDING 3NF SQLITE DATA MODEL ===

[STEP 1] Loading CSV into DataFrame…
[1] Loading heart.csv from local file…
Successfully loaded 1025 rows.
Loaded 1025 rows.

[STEP 2] Creating surrogate key patient_id…
patient_id added.

[STEP 3] Building dimension tables for categorical features…
Found 4 unique chest pain types.
Found 3 unique ECG results.
Found 3 unique slope values.
Found 4 unique thal values.

[STEP 4] Merging dimension IDs into main DataFrame…

[STEP 5] Creating 3NF DataFrames…
3NF DataFrames created.

[STEP 6] Creating SQLite database and tables…
Running SQL schema creation script…
Tables created.

[STEP 7] Inserting data into SQLite database…
Inserting chest_pain_type dimension…
Inserting ecg dimension…
Inserting slope dimension…
Inserting thal dimension…
Inserting patient table…
Inserting diagnosis table…

=== DONE! SQLite DB created at: ./data/heart.db ===

